<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/testing_skill_ckeck.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

GPU available: True
Sat Oct 17 10:43:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [2]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q "qwen-omni-utils[decord]" pymupdf python-docx pandas
# Optional (only if you set SHOW_EMBED = True below):
# !pip install -q sentence-transformers scikit-learn
# If the Qwen2.5-Omni import fails, use the dev build instead:
# !pip install -q git+https://github.com/huggingface/transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 93.9 MB/s eta 0:00:00


In [3]:
# [CHANGED] tries fp16 WITHOUT the speech modules first (4-bit quantization was hurting accuracy)
import os, gc, glob, re, time, hashlib, subprocess, math
from concurrent.futures import ThreadPoolExecutor
import torch, pymupdf, docx
import numpy as np, pandas as pd
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor, BitsAndBytesConfig
from qwen_omni_utils import process_mm_info

assert torch.cuda.is_available(), "No GPU: set the runtime type to T4 GPU and restart."

model_id = "Qwen/Qwen2.5-Omni-3B"
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"      # required for batched generation and last-token logits

def load(quantize_4bit=False):
    kwargs = dict(torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
    if quantize_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4",
            llm_int8_skip_modules=["audio_tower", "visual", "lm_head"])
    try:   # don't load the talker/token2wav (speech output): we only need text, and it lets fp16 fit on a T4
        return Qwen2_5OmniForConditionalGeneration.from_pretrained(model_id, enable_audio_output=False, **kwargs)
    except (TypeError, ValueError):
        return Qwen2_5OmniForConditionalGeneration.from_pretrained(model_id, **kwargs)

try:
    model = load(quantize_4bit=False); print("Loaded fp16")
except torch.cuda.OutOfMemoryError:
    gc.collect(); torch.cuda.empty_cache()
    model = load(quantize_4bit=True); print("fp16 didn't fit, loaded 4-bit")

if hasattr(model, "disable_talker"):      # text output only: free the speech modules
    model.disable_talker()
thinker = model.thinker
torch.cuda.empty_cache()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

chat_template.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

tokenizer_config.json:   0%|          | 0.00/6.47k [00:00<?, ?B/s]

...

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn.to_out.0.weight                            | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.ff.ff.{0, 3}.weight                             | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.k_proj.weight                                                     | UNEXPECTED |  | 
talker.model.layers.{0...23}.mlp.down_proj.weight                                                        | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn.to_v.weight                                | UNEXPECTED |  | 
token2wav.code2wav_dit_model.input_embed.s

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

spk_dict.pt: reconstructing file:   0%|          |  0.00B /  260kB            

spk_dict.pt: downloading bytes:           |  0.00B            

Loaded fp16
GPU memory used: 9.4 GB


In [4]:
# [CHANGED] added USE_AUDIO_IN_VIDEO, IMG_MIN_PX, IMG_MAX_PX
IMG = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
VID = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
AUD = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
TXT = {".txt", ".md", ".csv", ".json", ".html"}

# ---- text / PDF ----
MAX_CHARS, MAX_PIECES = 12000, 8       # long text: summarize up to MAX_PIECES pieces spread across the document

# ---- video ----
SHORT_VIDEO_SEC = 30      # videos up to this length are sent to the model whole
CLIP_SEC        = 10      # length of each sampled clip for longer videos
MAX_CLIPS       = 6       # at most this many clips per video (speed vs. coverage knob)
CLIP_HEIGHT     = 360     # clips are downscaled to this height before decoding

# ---- SPEED PRESET: "fast" | "balanced" | "quality" ----
# Visual tokens ~ (nframes / 2) * (max_pixels / 784). Prefill time grows with it.
#   quality  : 24 frames @ 360x420 -> ~2300 visual tokens  (V2 behaviour)
#   balanced : 12 frames @ 280x280 -> ~600
#   fast     :  8 frames @ 224x224 -> ~260
PRESET = "fast"
SUMMARY_SENTENCES = 1     # shorter generated summary = fewer decode steps (1 is fastest, 2-3 = more detail)
VIDEO_MODE = "single"     # "single": ONE model call per video, frames spread over the whole video (fastest)
                          # "clips" : describe up to MAX_CLIPS clips, then summarize the notes (slower, more temporal detail)
MAX_FRAME_MULT = 3        # in "single" mode, longer videos get up to this many times the preset's frame count
PRESCALE_SHORT = False    # True: downscale + 4 fps via ffmpeg before decoding (helps if profile shows slow decode_media)

USE_AUDIO_IN_VIDEO = False   # True = the model also LISTENS to the video's audio track (great for talking/tutorial videos;
                             # videos with no audio track are handled automatically). Costs extra tokens/time.
IMG_MIN_PX = 256 * 28 * 28   # images are resized into this range: big enough to read on-screen text, small enough for a T4
IMG_MAX_PX = 768 * 28 * 28

# Tiers from best to cheapest. On OOM the code steps down this list; STATE remembers what worked.
# min_pixels is set explicitly: newer qwen-omni-utils defaults it to ~100k pixels, which is larger
# than the max_pixels of the cheaper tiers and triggers "max_pixels must be >= min_pixels".
MIN_PX = 16 * 28 * 28
TIERS = [
    {"nframes": 24, "max_pixels": 360 * 420, "min_pixels": MIN_PX},   # quality
    {"nframes": 12, "max_pixels": 280 * 280, "min_pixels": MIN_PX},   # balanced
    {"nframes": 8,  "max_pixels": 224 * 224, "min_pixels": MIN_PX},   # fast
    {"nframes": 8,  "max_pixels": 168 * 168, "min_pixels": MIN_PX},   # emergency
]
_start = {"quality": 0, "balanced": 1, "fast": 2}[PRESET]
STATE = {"single_tier": _start, "clip_tier": max(_start, 1), "batch": 3}
FINAL_TOKENS = 30 + 30 * SUMMARY_SENTENCES   # output budget for TOPIC + SUMMARY

# ---- judge ----
JUDGE_THRESHOLD = 0.5     # a skill is matched when judge >= this. Tune it on a few labeled examples.
JUDGE_BATCH     = 8       # skills scored per forward pass (lower it if you hit OOM with many skills)
SHOW_EMBED      = False   # True adds an "embed" similarity column (needs sentence-transformers)

LABEL_EXAMPLES = "'Programming' (not 'Tower of Hanoi puzzle'), 'Chess' (not 'two players moving pieces'), 'Cooking', 'Dogs', 'Finance', 'Movies'"

SYSTEM = {"role": "system", "content": [{"type": "text",
    "text": "You are a concise assistant. Answer directly in plain text. Never ask the user questions."}]}

# One generation returns both the label and the summary
FINAL_PROMPT = ("Reply in exactly this format and nothing else:\n"
    "TOPIC: <1 or 2 words, the broad subject field, not a description. Examples: " + LABEL_EXAMPLES + ">\n"
    f"SUMMARY: <what this is about, at most {SUMMARY_SENTENCES} short sentences>")
CLIP_PROMPT = "Describe what happens in this clip in 2 short sentences."

In [5]:
# [CHANGED] _generate can use the video's audio track (USE_AUDIO_IN_VIDEO)
def _cleanup():
    gc.collect(); torch.cuda.empty_cache()

PROFILE = {}   # seconds per stage (plus token counts), reset at the start of every check()
def _add(k, v): PROFILE[k] = PROFILE.get(k, 0) + v

def has_audio(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=index",
                        "-of", "csv=p=0", path], capture_output=True, text=True)
    return bool(r.stdout.strip())

def _generate(convs, max_new_tokens):
    """Run a batch of conversations through the thinker. Returns one string per conversation."""
    t0 = time.time()
    texts = processor.apply_chat_template(convs, add_generation_prompt=True, tokenize=False)
    vids = [c["video"] for conv in convs for m in conv if isinstance(m["content"], list)
            for c in m["content"] if c.get("type") == "video"]
    uav = bool(USE_AUDIO_IN_VIDEO and vids and all(has_audio(v) for v in vids))   # listen to the video's sound too
    audios, images, videos = process_mm_info(convs, use_audio_in_video=uav)   # video decoding happens here
    t1 = time.time()
    inputs = processor(text=texts, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True, use_audio_in_video=uav)
    inputs = inputs.to("cuda").to(torch.float16)
    torch.cuda.synchronize(); t2 = time.time()
    with torch.inference_mode():
        out = thinker.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_audio_in_video=uav)
    torch.cuda.synchronize(); t3 = time.time()
    n_in = inputs.input_ids.shape[1]
    _add("decode_media_s", t1 - t0); _add("preprocess_s", t2 - t1); _add("prefill+generate_s", t3 - t2)
    _add("input_tokens", n_in * len(convs)); _add("new_tokens", (out.shape[1] - n_in) * len(convs))
    return [t.strip() for t in processor.batch_decode(out[:, n_in:], skip_special_tokens=True)]

def generate_many(contents, max_new_tokens=200, tier_key="single_tier"):
    """
    contents : list of content-part lists (one per request). Processed in batches of STATE["batch"].
    On OOM: first halve the batch size, then drop to a cheaper video tier. Both are remembered.
    """
    results, i = [None] * len(contents), 0
    while i < len(contents):
        group = contents[i:i + STATE["batch"]]
        has_video = any(c["type"] == "video" for content in group for c in content)
        convs = []
        for content in group:
            content = [dict(c) for c in content]
            for c in content:
                if c["type"] == "video":
                    m = c.pop("frame_mult", 1)                       # longer video -> more frames (still even)
                    c.update(TIERS[STATE[tier_key]])
                    c["nframes"] = max(4, int(round(c["nframes"] * m / 2)) * 2)
            convs.append([SYSTEM, {"role": "user", "content": content}])
        try:
            results[i:i + len(group)] = _generate(convs, max_new_tokens)
            i += len(group)
        except torch.cuda.OutOfMemoryError:
            _cleanup()
            if STATE["batch"] > 1:
                STATE["batch"] = max(1, STATE["batch"] // 2)
                print(f"Out of memory -> batch size now {STATE['batch']}")
            elif has_video and STATE[tier_key] < len(TIERS) - 1:
                STATE[tier_key] += 1
                print(f"Out of memory -> video tier now {TIERS[STATE[tier_key]]}")
            else:
                results[i] = "(out of memory)"; i += 1
    _cleanup()
    return results

def run_text(prompt, max_new_tokens=200):
    return generate_many([[{"type": "text", "text": prompt}]], max_new_tokens)[0]

# Warm-up: the first generate call pays one-time CUDA/kernel setup. Do it now so timings below are honest.
_t = time.time(); run_text("Say OK.", 4); PROFILE.clear()
print(f"Warm-up done in {time.time() - _t:.1f}s")

Warm-up done in 2.6s


In [6]:
# [CHANGED] images/screenshots are sent at a controlled, readable resolution
# ---- images, audio, PDF, docx, text ----
def _ext(src):
    """File extension for an existing path or a URL; '' for plain typed-in text."""
    if src.startswith("http") or os.path.exists(src):
        return os.path.splitext(src.split("?")[0])[1].lower()
    return ""

def _spread(items, k):
    """Pick up to k items evenly across the list (beginning, middle and end)."""
    if len(items) <= k: return items
    idx = np.unique(np.linspace(0, len(items) - 1, k).astype(int))
    return [items[i] for i in idx]

def _text_content(text):
    if len(text) > MAX_CHARS:             # long text: summarize pieces (batched), then the notes
        pieces = _spread([text[i:i + MAX_CHARS] for i in range(0, len(text), MAX_CHARS)], MAX_PIECES)
        notes = generate_many([[{"type": "text", "text": f"{p}\n\nSummarize this part in 3 sentences."}]
                               for p in pieces], 100)
        text = "\n".join(f"Part {i + 1}: {n}" for i, n in enumerate(notes))
    return [{"type": "text", "text": f"{text}\n\n{FINAL_PROMPT}"}]

def _pdf_content(path):
    with pymupdf.open(path) as doc:
        text = "\n".join(page.get_text() for page in doc)
        if len(text.strip()) > 200:
            return _text_content(text)
        content = []                       # scanned PDF: first 3 pages as images
        for i, page in enumerate(doc):
            if i >= 3: break
            png = f"/content/_page{i}.png"
            page.get_pixmap(dpi=100).save(png)
            content.append({"type": "image", "image": png, "min_pixels": IMG_MIN_PX, "max_pixels": IMG_MAX_PX})
    return content + [{"type": "text", "text": FINAL_PROMPT}]

def build_content(src):
    """Non-video input -> model-ready content parts."""
    ext = _ext(src)
    if not ext and not src.startswith("http"):
        return _text_content(src)          # plain text typed in directly
    if ext in IMG: return [{"type": "image", "image": src, "min_pixels": IMG_MIN_PX, "max_pixels": IMG_MAX_PX},
                           {"type": "text", "text": FINAL_PROMPT}]
    if ext in AUD: return [{"type": "audio", "audio": src}, {"type": "text", "text": FINAL_PROMPT}]
    if ext == ".pdf": return _pdf_content(src)
    if ext == ".docx": return _text_content("\n".join(p.text for p in docx.Document(src).paragraphs))
    if ext in TXT:
        with open(src, errors="ignore") as f: return _text_content(f.read())
    raise ValueError(f"Unsupported file type: {ext}")

In [7]:
# ---- video: sample a few clips, describe them in batches, then one final summary ----
def video_duration(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", path], capture_output=True, text=True)
    return float(r.stdout.strip())

def clip_starts(duration):
    """Start times of up to MAX_CLIPS windows of CLIP_SEC, spread evenly across the video."""
    if duration <= MAX_CLIPS * CLIP_SEC:                       # short enough: contiguous coverage
        return [k * CLIP_SEC for k in range(math.ceil(duration / CLIP_SEC))]
    return [round((duration - CLIP_SEC) * k / (MAX_CLIPS - 1), 2) for k in range(MAX_CLIPS)]

def extract_clips(path, duration, out_dir="/content/clips"):
    """Cut only the sampled windows, downscaled and re-encoded fast (so cuts are exact, decode is cheap)."""
    os.makedirs(out_dir, exist_ok=True)
    for f in glob.glob(f"{out_dir}/*.mp4"): os.remove(f)
    starts = clip_starts(duration)
    def cut(k_s):
        k, s = k_s
        out = f"{out_dir}/clip_{k:02d}.mp4"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(s), "-t", str(CLIP_SEC), "-i", path,
                        "-an", "-vf", f"scale=-2:{CLIP_HEIGHT}", "-c:v", "libx264", "-preset", "ultrafast",
                        "-crf", "30", out], check=True)
        return out, s
    with ThreadPoolExecutor(max_workers=4) as ex:
        return list(ex.map(cut, enumerate(starts)))

def prescale(path, out="/content/prescaled.mp4"):
    """Optional: shrink a short video (low height, 4 fps) so frame decoding is cheap."""
    t = time.time()
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-an",
                    "-vf", f"scale=-2:{CLIP_HEIGHT},fps=4", "-c:v", "libx264", "-preset", "ultrafast",
                    "-crf", "30", out], check=True)
    _add("prescale_s", time.time() - t)
    return out

def _mmss(sec): return f"{int(sec // 60)}:{int(sec % 60):02d}"

def summarize_video(src):
    """Returns raw 'TOPIC: ...\nSUMMARY: ...' text."""
    duration = video_duration(src)
    if VIDEO_MODE == "single" or duration <= SHORT_VIDEO_SEC:
        video = prescale(src) if PRESCALE_SHORT else src
        mult = min(MAX_FRAME_MULT, max(1.0, duration / SHORT_VIDEO_SEC))
        content = [{"type": "video", "video": video, "frame_mult": mult}, {"type": "text", "text": FINAL_PROMPT}]
        return generate_many([content], FINAL_TOKENS, "single_tier")[0]
    _t = time.time()
    clips = extract_clips(src, duration)
    _add("extract_clips_s", time.time() - _t)
    notes = generate_many([[{"type": "video", "video": p}, {"type": "text", "text": CLIP_PROMPT}]
                           for p, _ in clips], 70, "clip_tier")
    joined = "\n".join(f"[{_mmss(s)}] {n}" for (_, s), n in zip(clips, notes))
    prompt = ("These are notes on clips sampled evenly across one video, with their timestamps:\n\n"
              f"{joined}\n\n{FINAL_PROMPT}")
    return generate_many([[{"type": "text", "text": prompt}]], FINAL_TOKENS, "clip_tier")[0]

def summarize(src):
    """Any input -> raw 'TOPIC: ...\nSUMMARY: ...' text."""
    if _ext(src) in VID: return summarize_video(src)
    return generate_many([build_content(src)], FINAL_TOKENS)[0]

In [8]:
# [CHANGED v2] richer summary (+KEYWORDS), neutral label examples, fairer judge rule, weaker 'none' veto; SPEED: shorter output, 12-frame videos, batch 4
import string

# ---- settings used by this cell ----
SUMMARY_SENTENCES = 2         # SPEED: every output token costs ~0.05 s on a T4, so keep the description short
FINAL_TOKENS = 50 + 30 * SUMMARY_SENTENCES      # = 110 (TOPIC + SUMMARY + KEYWORDS normally use ~90)
STATE["single_tier"] = 1      # SPEED: videos use 12 frames @ 280x280 (0 = 24 frames @ 360x420: slower, more detail)
STATE["batch"] = 4            # SPEED: inputs described together; halves itself automatically on out-of-memory
NONE_VETO   = True            # use the "none of these" check
NONE_WEIGHT = 0.5             # was 1.0: the veto was too strong and crushed real matches (0 = ignore, 1 = full)
USE_CONTRAST = True           # second, reversed question to cancel the model's "yes" bias

# Examples deliberately NOT related to your skills or your data: the model copies them (it answered "Tower of Hanoi"
# for a post because that phrase was in the example).
LABEL_EXAMPLES = ("'Photography' (not 'camera on a table'), 'Gardening', 'Football', 'Cooking', "
                  "'Finance', 'Music', 'Fitness'")
FINAL_PROMPT = ("Base your answer only on what is literally visible or spoken: the people, objects, setting, "
    "app or website names, any text on screen, code, and the activity. Do not guess a topic from loose cues.\n"
    "Reply in exactly this format and nothing else:\n"
    "TOPIC: <1 to 3 words, the specific subject field, not a description. Examples: " + LABEL_EXAMPLES + ">\n"
    f"SUMMARY: <what this is about, at most {SUMMARY_SENTENCES} short sentences>\n"
    "KEYWORDS: <up to 6 literal words or names that are seen or heard: tools, apps, languages, commands, on-screen text>")

# ---- helpers ----
tok = processor.tokenizer
def _ids(words): return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})
YES_IDS, NO_IDS = _ids(["YES", "Yes", "yes"]), _ids(["NO", "No", "no"])
LETTERS = string.ascii_uppercase
LETTER_IDS = [tok.encode(c, add_special_tokens=False)[0] for c in LETTERS]

JUDGE_SYSTEM = {"role": "system", "content": [{"type": "text",
    "text": "You are a strict classifier. Reply with exactly one word: YES or NO."}]}
CHOICE_SYSTEM = {"role": "system", "content": [{"type": "text",
    "text": "You are a strict classifier. Reply with a single letter only."}]}

def skill_text(skill): return f"{skill['name']}: {skill['description']}"

def clean_label(raw, max_words=3):
    label = raw.strip().split("\n")[0].strip(" \"'`*.:;-<>")
    if label.lower().startswith("label"):
        label = label.split(":", 1)[-1].strip(" \"'`*.")
    return " ".join(label.split()[:max_words])

def parse_output(raw):
    t = re.search(r"TOPIC:\s*(.+)", raw, re.I)
    s = re.search(r"SUMMARY:\s*(.+)", raw, re.I | re.S)
    return (clean_label(t.group(1)) if t else ""), (s.group(1).strip() if s else raw.strip())

def make_label(summary):
    """Fallback only: used when the combined generation did not return a TOPIC line."""
    q = (f"Content summary:\n{summary}\n\nWhat general subject or field is this content about? "
         f"Answer with 1 or 2 words only: the broad category, not a description. Examples: {LABEL_EXAMPLES}. "
         "Output only the category.")
    return clean_label(run_text(q, 8))

def _last_logits(prompts):
    inputs = processor(text=prompts, return_tensors="pt", padding=True, use_audio_in_video=False)
    inputs = inputs.to("cuda").to(torch.float16)
    with torch.inference_mode():
        try:    out = thinker(**inputs, use_cache=False, logits_to_keep=1)
        except TypeError: out = thinker(**inputs, use_cache=False)
        logits = out.logits[:, -1].float()
    del out; return logits

def _p_yes(prompts):
    logits = _last_logits(prompts)
    yes, no = torch.logsumexp(logits[:, YES_IDS], 1), torch.logsumexp(logits[:, NO_IDS], 1)
    p = torch.sigmoid(yes - no).tolist()
    del logits; _cleanup(); return p

# ---- check A: strict YES/NO per skill (asked twice, in opposite directions) ----
JUDGE_RULE = ("Answer YES if the content is about this skill's subject area. That includes doing, practising, testing, "
              "building or learning it, and using tools, apps, code or commands that belong to it. Judge by what the "
              "content actually shows or says (the KEYWORDS count too), NOT by shared, ambiguous or look-alike words "
              "(a word can mean something different in another field). If the content is about a clearly different "
              "field, answer NO.")

def judge_all(label, summary, skills):
    """Per skill: average of P(YES: it is about the skill) and P(NO: it is about something different)."""
    probs = []
    for i in range(0, len(skills), JUDGE_BATCH):
        chunk = skills[i:i + JUDGE_BATCH]
        def build(question):
            ps = []
            for s in chunk:
                q = (f"Content topic: {label}\nContent summary: {summary}\n\nSkill: {skill_text(s)}\n\n"
                     f"{JUDGE_RULE}\n\n{question}")
                conv = [JUDGE_SYSTEM, {"role": "user", "content": [{"type": "text", "text": q}]}]
                ps.append(processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False))
            return ps
        p_in = _p_yes(build("Is the content clearly about this skill's subject area? Answer YES or NO."))
        if USE_CONTRAST:
            p_out = _p_yes(build("Is the content about something DIFFERENT from this skill's subject area, "
                                 "so that it does NOT belong to this skill? Answer YES or NO."))
            p_in = [(a + (1 - b)) / 2 for a, b in zip(p_in, p_out)]
        probs += p_in
    return probs

# ---- check B: pick ONE option, with "none of these" available ----
def none_prob(label, summary, skills):
    """P(the content belongs to none of the skills). Returns 0.0 if disabled or too many skills."""
    n = len(skills)
    if not NONE_VETO or n == 0 or n > len(LETTERS) - 1: return 0.0
    opts = "\n".join(f"{LETTERS[i]}. {skill_text(s)}" for i, s in enumerate(skills))
    opts += f"\n{LETTERS[n]}. None of these (the content is about something else)"
    q = (f"Content topic: {label}\nContent summary: {summary}\n\n"
         "Which ONE of these subject areas does the content mainly belong to? Judge by what the content is "
         "actually about, not by shared or look-alike words. If no option truly fits, choose the last one.\n\n"
         f"{opts}\n\nAnswer with the letter only.")
    conv = [CHOICE_SYSTEM, {"role": "user", "content": [{"type": "text", "text": q}]}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    logits = _last_logits([prompt])[0]
    p = torch.softmax(logits[LETTER_IDS[:n + 1]], 0)
    _cleanup()
    return float(p[n])

_embedder = None
def embed_scores(summary, skills):
    global _embedder
    from sentence_transformers import SentenceTransformer, util
    if _embedder is None:
        _embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    e_s = _embedder.encode(summary, convert_to_tensor=True, normalize_embeddings=True)
    e_k = _embedder.encode([skill_text(s) for s in skills], convert_to_tensor=True, normalize_embeddings=True)
    return util.cos_sim(e_s, e_k)[0].tolist()

# ---- describe + score (shared by check() below and by the post checker) ----
CACHE = {}   # (label, summary) per input; reset every time this cell is run
def _key(src):
    if os.path.exists(src):
        st = os.stat(src); return ("file", os.path.abspath(src), st.st_size, int(st.st_mtime))
    return ("str", hashlib.md5(src.encode()).hexdigest())

def describe(src, use_cache=True):
    """Any input (text, image, video, audio, pdf, docx...) -> (label, summary)."""
    key = _key(src)
    if use_cache and key in CACHE: return CACHE[key]
    label, summary = parse_output(summarize(src))
    if not label: label = make_label(summary)
    CACHE[key] = (label, summary)
    return label, summary

def score_skills(label, summary, skills, threshold=None):
    """(label, summary) + skills -> (table, p_none). Final score 'judge' = yes_score * (1 - NONE_WEIGHT * p_none)."""
    threshold = JUDGE_THRESHOLD if threshold is None else threshold
    yes = judge_all(label, summary, skills)
    p_none = none_prob(label, summary, skills)
    rows = [{"skill": s["name"], "yes": round(y, 3), "judge": round(y * (1 - NONE_WEIGHT * p_none), 3)}
            for s, y in zip(skills, yes)]
    table = pd.DataFrame(rows)
    if SHOW_EMBED: table["embed"] = [round(x, 3) for x in embed_scores(summary, skills)]
    table = table.sort_values("judge", ascending=False).reset_index(drop=True)
    table["matched"] = table.judge >= threshold
    return table, p_none

def check(src, skills, threshold=None, use_cache=True):
    """Same return shape as before, plus 'p_none'."""
    timing = {}
    PROFILE.clear()
    t = time.time(); cached = use_cache and _key(src) in CACHE
    label, summary = describe(src, use_cache)
    timing["summarize"] = 0.0 if cached else round(time.time() - t, 1)
    t = time.time()
    table, p_none = score_skills(label, summary, skills, threshold)
    timing["judge"] = round(time.time() - t, 1)
    matches = table[table.matched].skill.tolist()
    return {"related": bool(matches), "matches": matches, "label": label, "p_none": round(p_none, 3),
            "summary": summary, "table": table, "timing": timing,
            "profile": {k: round(v, 2) for k, v in PROFILE.items()}}

def show(r, verbose=False):
    print(f"Topic   : {r['label']}   (P none-of-these: {r.get('p_none')})")
    print(f"Related : {r['related']}   Matched: {r['matches']}")
    print(f"Time (s): {r['timing']}")
    if r.get("profile"): print(f"Profile : {r['profile']}")
    if verbose: print(f"Summary : {r['summary']}")
    return r["table"]

## Real data from the backend


In [9]:
# [CHANGED] much richer skill descriptions (the judge only ever sees these) - EDIT them to match what each skill really covers
# ===== REAL DATA: user + skills + posts from the backend =====
import requests, json, os
from urllib.parse import urlparse, urljoin

API_BASE  = "https://evolve-backend-4lux.onrender.com"
USER_ID   = 1
ENDPOINT  = "/api/skills/user/{user_id}/skills-posts"
ALLOW_MOCK_FALLBACK = False         # real API: show the real error instead of silently using fake data
REQUEST_TIMEOUT = 90                # Render free instances sleep; the first request can take ~1 minute
MEDIA_DIR = "/content/media"

# Login token: Colab sidebar -> key icon (Secrets) -> name EVOLVE_TOKEN, notebook access ON.
# No secret = hidden prompt. Never write the token inside the notebook.
API_TOKEN = None
try:
    from google.colab import userdata
    API_TOKEN = userdata.get("EVOLVE_TOKEN")
except Exception:
    pass
if not API_TOKEN:
    from getpass import getpass
    API_TOKEN = getpass("Paste API token (leave empty if none): ")
# clean up copy/paste leftovers: spaces, line breaks, quotes, a leading "Bearer "
if API_TOKEN:
    API_TOKEN = API_TOKEN.strip().strip("\"'").strip()
    if API_TOKEN.lower().startswith("bearer "): API_TOKEN = API_TOKEN[7:].strip()
    API_TOKEN = API_TOKEN or None

# The API returns "description": null for every skill, and the judge compares posts against descriptions.
# Until the backend fills them in, these are used when the API has none (matched by skill name).
SKILL_DESCRIPTIONS = {
    "Python":          "Python programming: writing, running and debugging Python code, scripts, functions, loops, classes, "
                       "pip and libraries (pandas, numpy, Flask, Django, FastAPI), Jupyter/Colab notebooks, automation, Python tutorials",
    "AI/ML":           "artificial intelligence and machine learning: ML models, neural networks, deep learning, training and "
                       "evaluating models, datasets, data science, computer vision, NLP, LLMs, prompts, PyTorch, TensorFlow, Hugging Face",
    "Web Development": "building websites and web apps: HTML, CSS, JavaScript, TypeScript, React, Node.js, frontend and backend, "
                       "REST APIs and API testing (Postman, endpoints, requests and responses), login and authentication screens, "
                       "forms, databases, UI pages, web project code and Git",
    "Cloud":           "cloud computing and DevOps: AWS, Azure, Google Cloud, servers, hosting and deployment, Docker, Kubernetes, "
                       "CI/CD pipelines, networking, cloud storage and infrastructure",
}

# Shape of the real response (trimmed). Only used if you set ALLOW_MOCK_FALLBACK = True.
MOCK_RESPONSE = {
    "success": True, "userId": 1, "user": {"id": 1, "name": "a_mor"},
    "skills": [
        {"id": 1, "skillId": 1, "relationshipType": "LEARNING", "skill": {"id": 1, "name": "Python", "description": None, "isActive": True}},
        {"id": 2, "skillId": 3, "relationshipType": "LEARNING", "skill": {"id": 3, "name": "AI/ML", "description": None, "isActive": True}},
    ],
    "posts": [{"id": 1, "caption": "My first EVOLV post", "postType": "IMAGE", "status": "PUBLISHED", "deletedAt": None,
               "media": [{"id": 1, "mediaType": "IMAGE", "mediaUrl": "https://example.com/post_1.png", "position": 0}]}],
}

def _first(d, keys, default=""):
    for k in keys:
        v = d.get(k)
        if v not in (None, ""): return v
    return default

def _headers():
    h = {}
    if API_TOKEN: h["Authorization"] = f"Bearer {API_TOKEN}"
    return h

def fetch_user_data(user_id=USER_ID, allow_mock=None):
    allow_mock = ALLOW_MOCK_FALLBACK if allow_mock is None else allow_mock
    url = API_BASE.rstrip("/") + ENDPOINT.format(user_id=user_id)
    try:
        r = requests.get(url, headers=_headers(), timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        data = r.json()
        print(f"Fetched from {url}")
    except Exception as e:
        status = getattr(getattr(e, "response", None), "status_code", None)
        if status in (401, 403): print(f"HTTP {status}: the API needs a valid login token (see API_TOKEN above).")
        if not allow_mock: raise
        print(f"Could not reach {url}\n   {type(e).__name__}: {e}\n   -> using MOCK_RESPONSE instead")
        data = MOCK_RESPONSE
    return data.get("data", data) if isinstance(data, dict) else data

def normalize(data):
    """API JSON -> (user, skills, posts).
    skills: [{id, name, description}]            (the API nests each skill under "skill")
    posts : [{id, text, media:[{id,url,type}]}]  (caption is the text; media is a list, in order)"""
    user = data.get("user") or {"id": data.get("userId", USER_ID)}
    skills = []
    for i, us in enumerate(data.get("skills", [])):
        s = us.get("skill") or us
        if s.get("isActive") is False: continue
        name = _first(s, ["name", "title", "skill_name"])
        skills.append({"id": s.get("id", us.get("skillId", i)), "name": name,
                       "description": _first(s, ["description", "desc", "details"]) or SKILL_DESCRIPTIONS.get(name, name)})
    posts = []
    for i, p in enumerate(data.get("posts", [])):
        if p.get("deletedAt"): continue
        text = "\n".join(x for x in [_first(p, ["title"]), _first(p, ["caption", "content", "text", "body", "description"])] if x)
        media = [{"id": m.get("id"), "url": _first(m, ["mediaUrl", "media_url", "url"], None), "type": m.get("mediaType")}
                 for m in sorted(p.get("media") or [], key=lambda m: m.get("position") or 0)]
        posts.append({"id": p.get("id", i), "text": text, "media": [m for m in media if m["url"]], "type": p.get("postType")})
    return user, skills, posts

def get_media(m, post_id):
    """Download one media item to a local file (the model code needs a local path for image/video)."""
    url = m["url"]
    if not url.startswith("http"): url = urljoin(API_BASE.rstrip("/") + "/", url.lstrip("/"))
    ext = os.path.splitext(urlparse(url).path)[1].lower() or {"IMAGE": ".jpg", "VIDEO": ".mp4"}.get((m.get("type") or "").upper(), "")
    os.makedirs(MEDIA_DIR, exist_ok=True)
    path = f"{MEDIA_DIR}/post_{post_id}_media_{m.get('id')}{ext}"
    if not os.path.exists(path):
        # send the login token ONLY to our own backend, never to Cloudinary or other hosts
        headers = _headers() if urlparse(url).netloc == urlparse(API_BASE).netloc else {}
        with requests.get(url, headers=headers, stream=True, timeout=REQUEST_TIMEOUT) as r:
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(1 << 20): f.write(chunk)
    return path


In [10]:
# [CHANGED v2] short captions (2+ words) are used too, placed first; SPEED: all posts are described in batches (check_posts)
user, SKILLS, POSTS = normalize(fetch_user_data())
print(f"User {user.get('id')} ({user.get('name', '')}) | {len(SKILLS)} skills: {[s['name'] for s in SKILLS]} | {len(POSTS)} posts")

def check_post(post, skills, threshold=None):
    """One post (+ that user's skills) -> JSON-friendly dict.
    Every media item and the caption are first DESCRIBED, then everything is joined and JUDGED ONCE.
    (Old version judged each part alone and kept the best score = every extra part was another chance
    for a false match.) A failure becomes 'error'."""
    skill_id = {s["name"]: s["id"] for s in skills}
    labels, summaries, errors = [], [], []
    for m in post["media"]:
        try:
            label, summary = describe(get_media(m, post["id"]))
            labels.append(label); summaries.append(f"[{(m.get('type') or 'media').lower()}] {summary}")
        except Exception as e:
            errors.append(f"media: {type(e).__name__}: {e}")
    text = post["text"].strip()
    if text:
        if post["media"]:
            if len(text.split()) >= 2: summaries.insert(0, f"[caption written by the author] {text[:500]}")   # caption first: strongest clue
        else:
            try:
                label, summary = describe(text)
                labels.append(label); summaries.append(f"[text] {summary}")
            except Exception as e:
                errors.append(f"text: {type(e).__name__}: {e}")
    if not summaries:
        return {"post_id": post["id"], "error": "; ".join(errors) or "post has no text or media"}
    label = " | ".join(dict.fromkeys(labels)) or "post"
    summary = " ".join(summaries)
    table, _ = score_skills(label, summary, skills, threshold)
    matches = table[table.matched].skill.tolist()
    out = {"post_id": post["id"], "topic": label, "summary": summary, "related": bool(matches),
           "matches": [{"skill_id": skill_id[n], "skill": n} for n in matches],
           "scores":  [{"skill_id": skill_id[r.skill], "skill": r.skill, "score": float(r.judge)}
                       for r in table.itertuples()]}
    if errors: out["warnings"] = errors
    return out

# ---- SPEED: describe many inputs at once. Text generation on a GPU is memory-bound, so decoding 4 inputs together costs
# little more than decoding 1. Results are stored in CACHE, so check_post() below finds them already done. ----
def _group(src):
    e = _ext(src)
    return "video" if e in VID else "image" if e in IMG else "other"

def describe_many(srcs, use_cache=True):
    found, todo = {}, []
    for s in dict.fromkeys(srcs):                                  # unique, keep order
        if use_cache and _key(s) in CACHE: found[s] = CACHE[_key(s)]
        else: todo.append(s)
    groups = {"other": [], "image": [], "video": []}
    for s in todo:
        g = _group(s)
        if g == "video":
            d = video_duration(s)
            if VIDEO_MODE != "single" and d > SHORT_VIDEO_SEC:     # "clips" mode keeps its own (slower) path
                found[s] = describe(s, use_cache); continue
            v = prescale(s) if PRESCALE_SHORT else s
            content = [{"type": "video", "video": v, "frame_mult": min(MAX_FRAME_MULT, max(1.0, d / SHORT_VIDEO_SEC))},
                       {"type": "text", "text": FINAL_PROMPT}]
        else:
            content = build_content(s)
        groups[g].append((s, content))
    for g in ("other", "image", "video"):                          # never mix kinds in one batch
        items = groups[g]
        if not items: continue
        raws = generate_many([c for _, c in items], FINAL_TOKENS, "single_tier")
        for (s, _), raw in zip(items, raws):
            label, summary = parse_output(raw)
            if not label: label = make_label(summary)
            CACHE[_key(s)] = found[s] = (label, summary)
    return found

def check_posts(posts, skills, threshold=None):
    t0 = time.time()
    srcs = []
    for p in posts:
        for m in p["media"]:
            try: srcs.append(get_media(m, p["id"]))
            except Exception: pass                                 # check_post() reports the download error
        if not p["media"] and p["text"].strip(): srcs.append(p["text"].strip())
    t1 = time.time()
    try: describe_many(srcs)
    except Exception as e: print("batched describe failed, falling back to one post at a time:", type(e).__name__, e)
    t2 = time.time()
    results = [check_post(p, skills, threshold) for p in posts]    # describe() is a cache hit now; this is the judging
    print(f"download {t1 - t0:.0f}s | describe {t2 - t1:.0f}s | judge {time.time() - t2:.0f}s | "
          f"total {time.time() - t0:.0f}s for {len(posts)} posts")
    return results

RESULTS = check_posts(POSTS, SKILLS)

display(pd.DataFrame([{"post_id": r["post_id"], "topic": r.get("topic", ""), "related": r.get("related"),
                       "matched_skills": ", ".join(m["skill"] for m in r.get("matches", [])),
                       "error": r.get("error", "")} for r in RESULTS]))

payload = {"user_id": user.get("id"), "results": RESULTS}      # the shape you can later POST back to the backend
with open("/content/skill_check_results.json", "w") as f: json.dump(payload, f, indent=2)
print("Saved /content/skill_check_results.json")

Fetched from https://evolve-backend-4lux.onrender.com/api/skills/user/1/skills-posts
User 1 (a_mor) | 4 skills: ['Python', 'AI/ML', 'Web Development', 'Cloud'] | 23 posts


qwen-vl-utils using torchcodec to read video.
[transformers] Qwen2VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


download 6s | describe 55s | judge 48s | total 109s for 23 posts


,post_id,topic,related,matched_skills,error
0,1,Login Page,True,Web Development,
1,2,Fitness,False,,
2,3,Software Development,True,"Cloud, Web Development",
3,4,API Testing,True,"Web Development, Cloud",
4,5,API Testing,True,Web Development,
5,9,Fitness,False,,
6,10,Fitness,False,,
7,13,Tower of Hanoi,False,,
8,14,Python Code Execution,True,"Python, AI/ML, Cloud",
9,15,Cloud Computing,True,Cloud,


Saved /content/skill_check_results.json


In [11]:
print(json.dumps(fetch_user_data(), indent=2)[:4000])

Fetched from https://evolve-backend-4lux.onrender.com/api/skills/user/1/skills-posts
{
  "success": true,
  "userId": 1,
  "user": {
    "id": 1,
    "name": "a_mor",
    "profile": {
      "ageRange": "19-22",
      "avatarUrl": "https://res.cloudinary.com/dh3de6exr/image/upload/v1792069055/evolv/avatars/user_1_1792069054481.png",
      "bio": "Full-stack developer",
      "countryCode": "IN",
      "createdAt": "2026-10-15T12:57:35.955225Z",
      "id": 1,
      "updatedAt": "2026-10-15T12:57:35.955225Z",
      "userId": 1,
      "username": "a_mor"
    }
  },
  "skills": [
    {
      "createdAt": "2026-10-15T13:05:13.292105Z",
      "id": 1,
      "relationshipType": "LEARNING",
      "skillId": 1,
      "updatedAt": "2026-10-15T13:05:13.292105Z",
      "userId": 1,
      "skill": {
        "categoryId": 1,
        "createdAt": "2026-10-15T12:54:04.317002Z",
        "description": null,
        "id": 1,
        "isActive": true,
        "name": "Python",
        "slug": "python",
 

## Host the model (API server on Colab)

In [12]:
!pip install -q fastapi uvicorn requests nest_asyncio

In [13]:
# ===== API server around the loaded model =====
# Endpoints (all need header  X-Webhook-Secret):
#   GET  /health                 -> status (no secret needed)
#   POST /webhook/new-post       -> {"user_id":1,"post_id":5}  queued, answers instantly
#   POST /check                  -> {"user_id":1,"post_id":5}  waits and returns the result
#   GET  /results                -> everything processed so far
import queue, threading, uvicorn
from fastapi import FastAPI, Header, HTTPException, Request

# Put the secret in Colab Secrets as WEBHOOK_SECRET (key icon, notebook access ON). Never hard-code it.
try:
    from google.colab import userdata
    WEBHOOK_SECRET = userdata.get("WEBHOOK_SECRET")
except Exception:
    WEBHOOK_SECRET = None
if not WEBHOOK_SECRET:
    from getpass import getpass
    WEBHOOK_SECRET = getpass("Choose a webhook secret (long random string): ")

SEND_RESULTS_BACK = False   # True once the backend has an endpoint that accepts the result
RESULT_ENDPOINT   = "/api/skills/user/{user_id}/posts/{post_id}/skill-match"   # PROPOSED, agree with backend team

JOBS = queue.Queue()                 # one GPU = one worker: jobs wait in line here
GPU_LOCK = threading.Lock()          # /check and the worker never use the model at the same time
RESULTS_BY_POST = {}
app = FastAPI()

def _auth(secret):
    if secret != WEBHOOK_SECRET: raise HTTPException(401, "bad secret")

def process_job(uid, pid):
    user, skills, posts = normalize(fetch_user_data(uid, allow_mock=False))
    post = next((p for p in posts if str(p["id"]) == str(pid)), None)
    if post is None: return {"post_id": pid, "error": "post not found in the API response"}
    with GPU_LOCK:
        return check_post(post, skills)

def send_back(uid, pid, res):
    url = API_BASE.rstrip("/") + RESULT_ENDPOINT.format(user_id=uid, post_id=pid)
    try: requests.post(url, json=res, headers=_headers(), timeout=REQUEST_TIMEOUT).raise_for_status()
    except Exception as e: print("send_back failed:", e)

def worker():
    while True:
        uid, pid = JOBS.get()
        try: res = process_job(uid, pid)
        except Exception as e: res = {"post_id": pid, "error": f"{type(e).__name__}: {e}"}
        RESULTS_BY_POST[(str(uid), str(pid))] = res
        print(f"[user {uid} | post {pid}] ->", res.get("matches", res.get("error")))
        if SEND_RESULTS_BACK and "error" not in res: send_back(uid, pid, res)
        JOBS.task_done()

@app.get("/health")
def health():
    return {"ok": True, "queued": JOBS.qsize(), "done": len(RESULTS_BY_POST)}

@app.get("/results")
def results(x_webhook_secret: str = Header(default="")):
    _auth(x_webhook_secret)
    return [{"user_id": u, "post_id": p, **r} for (u, p), r in RESULTS_BY_POST.items()]

async def _ids_from(request):
    body = await request.json()
    uid = body.get("user_id")
    pid = body.get("post_id") or (body.get("post") or {}).get("id")
    if uid is None or pid is None: raise HTTPException(422, "need user_id and post_id")
    return uid, pid

@app.post("/webhook/new-post")
async def new_post(request: Request, x_webhook_secret: str = Header(default="")):
    _auth(x_webhook_secret)
    uid, pid = await _ids_from(request)
    JOBS.put((uid, pid))
    return {"queued": True}

@app.post("/check")
def check_now(body: dict, x_webhook_secret: str = Header(default="")):
    _auth(x_webhook_secret)
    uid, pid = body.get("user_id"), body.get("post_id")
    if uid is None or pid is None: raise HTTPException(422, "need user_id and post_id")
    res = process_job(uid, pid)
    RESULTS_BY_POST[(str(uid), str(pid))] = res
    return res
print("API defined.")

Choose a webhook secret (long random string): ··········
API defined.


In [14]:
# ===== Start the server + public URL (run once per Colab session) =====
import subprocess, re, os
PORT = 8000
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
threading.Thread(target=worker, daemon=True).start()

if not os.path.exists("cloudflared"):   # free quick tunnel, no account needed
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
                   " -O cloudflared && chmod +x cloudflared", shell=True, check=True)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
                          stderr=subprocess.PIPE, text=True)
PUBLIC_URL = None
for line in tunnel.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: PUBLIC_URL = m.group(0); break
threading.Thread(target=lambda: [None for _ in tunnel.stderr], daemon=True).start()
assert PUBLIC_URL, "cloudflared did not give a public URL"

print("Health check :", PUBLIC_URL + "/health")
print("Webhook URL  :", PUBLIC_URL + "/webhook/new-post")
print("Sync check   :", PUBLIC_URL + "/check   (POST {user_id, post_id})")
print("Header       : X-Webhook-Secret: <your secret>")
print("The URL changes every time this cell is re-run.")

Health check : https://chrome-shore-ruth-enhance.trycloudflare.com/health
Webhook URL  : https://chrome-shore-ruth-enhance.trycloudflare.com/webhook/new-post
Sync check   : https://chrome-shore-ruth-enhance.trycloudflare.com/check   (POST {user_id, post_id})
Header       : X-Webhook-Secret: <your secret>
The URL changes every time this cell is re-run.


In [19]:
import requests

url = "https://chrome-shore-ruth-enhance.trycloudflare.com/check"
r = requests.post(
    url,
    headers={"X-Webhook-Secret":"evolve-k29x-8fj3-qp71-zt64"},
    json={"user_id": 1, "post_id": 2},
    timeout=300,
)
print(r.status_code)
print(r.json())

Fetched from https://evolve-backend-4lux.onrender.com/api/skills/user/1/skills-posts
200
{'post_id': 2, 'topic': 'Fitness', 'summary': '[caption written by the author] My first video post [video] A man is exercising on a rowing machine in a well-lit gym. He is wearing a yellow shirt and black shorts.\nKEYWORDS: Rowing machine, gym, workout, fitness, exercise, man', 'related': False, 'matches': [], 'scores': [{'skill_id': 3, 'skill': 'AI/ML', 'score': 0.244}, {'skill_id': 1, 'skill': 'Python', 'score': 0.14}, {'skill_id': 7, 'skill': 'Web Development', 'score': 0.107}, {'skill_id': 5, 'skill': 'Cloud', 'score': 0.082}]}
